In [ ]:
import pandas as pd
import nflreadpy as nfl
from pathlib import Path


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

schedule = nfl.load_schedules(seasons=2025)
schedule_df = schedule.to_pandas()
schedule_df.infer_objects(inplace=True)
schedule_df.info()
schedule_df.describe()

print(schedule_df.head())
print(schedule_df.columns)

           game_id  season game_type  week     gameday   weekday gametime  \
0  2025_01_DAL_PHI    2025       REG     1  2025-09-04  Thursday    20:20   
1   2025_01_KC_LAC    2025       REG     1  2025-09-05    Friday    20:00   
2   2025_01_TB_ATL    2025       REG     1  2025-09-07    Sunday    13:00   
3  2025_01_CIN_CLE    2025       REG     1  2025-09-07    Sunday    13:00   
4  2025_01_MIA_IND    2025       REG     1  2025-09-07    Sunday    13:00   

  away_team  away_score home_team  home_score location  result  total  \
0       DAL        20.0       PHI        24.0     Home     4.0   44.0   
1        KC        21.0       LAC        27.0  Neutral     6.0   48.0   
2        TB        23.0       ATL        20.0     Home    -3.0   43.0   
3       CIN        17.0       CLE        16.0     Home    -1.0   33.0   
4       MIA         8.0       IND        33.0     Home    25.0   41.0   

   overtime old_game_id     gsis nfl_detail_id           pfr    pff  \
0       0.0  2025090400  59

In [ ]:
logos = pd.read_csv("C:\\Users\\goku\\Documents\\NFL_ML_Predictions\\backend\\team_logo.csv")
print(logos.head())
print(logos.columns)
print(logos.info())
print(logos['team_abbr'].unique())

In [ ]:
import logging
from datetime import datetime, timezone, timedelta

def get_schedule() -> pd.DataFrame:
    """
    Return the schedule for the "next" NFL week based on the schedule CSV.

    Logic:
      - Resolve schedule path via _find_schedule_path().
      - Normalize season/week + team abbreviations.
      - If 'gameday' is present, interpret as kickoff datetime (UTC-aware).
      - Determine the next slate using the earliest future game; fall back
        to the latest season/week in the file if all games are in the past.
    """
    df_schedule = nfl.load_schedules().to_pandas()

    logging.info("[Schedule] Loading schedule from: %s", df_schedule)
    dfs = df_schedule.copy()

    now = datetime.now(timezone.utc)

    current_games = dfs[
        (pd.to_datetime(dfs["gameday"], errors="coerce", utc=True) >= now) & (pd.to_datetime(dfs["gameday"], errors="coerce", utc=True) < now + timedelta(days=7))
    ]
    logos_df = pd.read_csv("C:\\Users\\goku\\Documents\\NFL_ML_Predictions\\backend\\team_logo.csv")
    current_games = current_games.merge(logos_df, how="left", left_on="home_team")
    current_games = current_games.merge(logos_df, how="left", left_on="away_team", suffixes=('_home', '_away'))
    results = []
    results.append(
            {   "game_day": current_games.iloc[0]["gameday"],
                "game_id": f"{current_games.iloc[0]['season']}_{current_games.iloc[0]['week']}_{current_games.iloc[0]['home_team']}_{current_games.iloc[0]['away_team']}",
                "season": int(current_games.iloc[0]['season']),
                "week": int(current_games.iloc[0]['week']),
                "home_team_logo": current_games.iloc[0]['logo_home'],
                "away_team_logo": current_games.iloc[0]['logo_away'],
            }
    )
    return pd.DataFrame(results)

results = get_schedule()
print(results)